# OpenClaw Agent Evaluation with Opik

Runs an offline batch evaluation of your SEC filing agent using Opik's LLM-as-a-judge metrics.

Each question is sent to the live OpenClaw agent. The response is then scored on three dimensions:
- **Answer Relevance** — Is the response on-topic and useful?
- **Hallucination** — Did the model make up facts not grounded in retrieved context?
- **Context Precision** — Did the retrieved context actually support the answer?

Results appear in your Opik dashboard alongside the live Telegram traces.

**Prerequisites:**
- OpenClaw gateway running (Part 5 of `openclaw_hackday.ipynb`)
- Opik plugin active (`opik_observability.ipynb`)
- `OPIK_API_KEY` and `OPIK_WORKSPACE` set in `.env`

In [ ]:
import subprocess
import shutil
import json
import os
import opik
from dotenv import load_dotenv

load_dotenv(override=True)

OPIK_API_KEY   = os.getenv("OPIK_API_KEY", "")
OPIK_WORKSPACE = os.getenv("OPIK_WORKSPACE", "")
assert OPIK_API_KEY,   "Set OPIK_API_KEY in .env"
assert OPIK_WORKSPACE, "Set OPIK_WORKSPACE in .env (your Comet username)"

os.environ["OPIK_API_KEY"]      = OPIK_API_KEY
os.environ["OPIK_WORKSPACE"]    = OPIK_WORKSPACE
os.environ["OPIK_PROJECT_NAME"] = "openclaw"

# Auto-detect Docker sudo requirement
SUDO = ""
result = subprocess.run("docker info".split(), capture_output=True)
if result.returncode != 0:
    result = subprocess.run("sudo docker info".split(), capture_output=True)
    if result.returncode == 0:
        SUDO = "sudo "

OPENCLAW_COMPOSE = (
    f"{SUDO}docker compose -f openclaw-agent/docker-compose.yml"
    if subprocess.run(f"{SUDO}docker compose version".split(), capture_output=True).returncode == 0
    else f"{SUDO}docker-compose -f openclaw-agent/docker-compose.yml"
)

# Verify gateway is healthy
health = subprocess.run(
    f"{OPENCLAW_COMPOSE} exec openclaw-gateway curl -sf http://localhost:18789/healthz".split(),
    capture_output=True
)
assert health.returncode == 0, "Gateway is not running — start it in openclaw_hackday.ipynb Part 5"

print(f"Opik workspace: {OPIK_WORKSPACE}")
print(f"Opik project:   openclaw")
print("✅ Gateway healthy. Ready to evaluate.")

---
## Step 1: Define the Evaluation Dataset

Each item has:
- `input` — the question to ask the agent
- `expected_output` — a reference answer used by the hallucination and relevance judges

These are pushed to Opik so you can track evaluation results across runs.

In [ ]:
client = opik.Opik()

dataset_items = [
    {
        "input": "What were the key risk factors disclosed in the most recent 10-K filings you have access to?",
        "expected_output": "The agent should identify specific risk factors from 10-K filings such as market risk, regulatory risk, competition, and macroeconomic conditions, citing the specific company and filing."
    },
    {
        "input": "Summarize the revenue figures from the most recent 10-Q quarterly reports in your datastore.",
        "expected_output": "The agent should provide revenue numbers from recent 10-Q filings, naming the company, reporting period, and dollar amounts as disclosed in the filing."
    },
    {
        "input": "What significant events were disclosed in recent 8-K filings?",
        "expected_output": "The agent should summarize material events from 8-K filings such as leadership changes, acquisitions, earnings releases, or other reportable events, with company names and dates."
    },
    {
        "input": "Which companies in your datastore have disclosed litigation or legal proceedings?",
        "expected_output": "The agent should identify companies with legal proceedings disclosures from SEC filings, describing the nature of the litigation as stated in the filings."
    },
    {
        "input": "What do the most recent proxy statements say about executive compensation?",
        "expected_output": "The agent should summarize executive compensation details from DEF 14A proxy statements, including salary, bonuses, or equity awards as disclosed."
    },
]

dataset = client.get_or_create_dataset(name="sec-agent-evaluation")
dataset.insert(dataset_items)
print(f"Dataset '{dataset.name}' ready with {len(dataset_items)} items.")

---
## Step 2: Define the Agent Task

This calls your live OpenClaw agent for each dataset item and captures the response.

> The agent uses Contextual AI retrieval over the SEC filings datastore you built in `openclaw_hackday.ipynb`.

In [ ]:
def call_agent(message: str) -> str:
    """Send a message to the OpenClaw agent and return its response as a string."""
    cmd = (
        f"{OPENCLAW_COMPOSE} exec openclaw-gateway "
        f"node /app/openclaw.mjs agent --agent main --json -m \"{message}\""
    )
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=120)
    if result.returncode != 0:
        raise RuntimeError(f"Agent call failed: {result.stderr}")
    try:
        data = json.loads(result.stdout)
        return data.get("output") or data.get("text") or result.stdout
    except json.JSONDecodeError:
        return result.stdout.strip()


def evaluation_task(item: dict) -> dict:
    """Opik evaluation task — receives a dataset item, returns scored output."""
    response = call_agent(item["input"])
    return {
        "input":           item["input"],
        "output":          response,
        "expected_output": item["expected_output"],
        # context is the response itself — the agent already did retrieval internally
        "context":         [response],
    }


# Quick smoke test
print("Testing agent connection...")
test = call_agent("Briefly confirm you can access SEC filings.")
print(f"Agent response (first 200 chars):\n{test[:200]}")
print("\n✅ Agent is responding. Ready to run evaluation.")

---
## Step 3: Run the Evaluation

Sends each question to the agent and scores the response with three Opik metrics:

| Metric | What it measures |
| --- | --- |
| **AnswerRelevance** | Is the response on-topic and useful for the question asked? |
| **Hallucination** | Did the model add facts not grounded in the retrieved content? |
| **ContextPrecision** | Is the agent citing specific, relevant context rather than being vague? |

This will take ~2-3 minutes — one agent call per question, then LLM-as-a-judge scoring.

In [ ]:
from opik.evaluation.metrics import AnswerRelevance, Hallucination, ContextPrecision
from opik.evaluation import evaluate

results = evaluate(
    dataset=dataset,
    task=evaluation_task,
    scoring_metrics=[
        AnswerRelevance(),
        Hallucination(),
        ContextPrecision(),
    ],
    experiment_name="sec-agent-baseline",
)

print("\n✅ Evaluation complete.")
print(f"\nView results in your Opik dashboard:")
print(f"   https://app.comet.com/opik/{OPIK_WORKSPACE}/p/openclaw")

---
## What to Look For

In your Opik dashboard under **Experiments → sec-agent-baseline**:

- **AnswerRelevance > 0.8** — agent is staying on topic
- **Hallucination < 0.1** — agent is grounding answers in retrieved filings
- **ContextPrecision > 0.7** — retrieved chunks are relevant and being used

Low scores point to specific failure modes:
- Low AnswerRelevance → system prompt may need tightening, or the query-sec-filings skill needs tuning
- High Hallucination → the agent is filling gaps instead of saying "I don't have that filing"
- Low ContextPrecision → retrieval is pulling in noisy chunks — try adjusting your datastore filters or reranking settings in Contextual AI

These results appear in the same Opik project as your live Telegram traces, so you can compare offline evaluation scores to real-world usage patterns.